## this notebook is used to plot cohort concordance and GWAS results from the "Cohort Builder"-derived cohort and SQL-derived cohort

adapted from notebook:

"5 - GWAS Figures 2024.ipynb"

in workspace "Hypothyroidism genomics v7"

In [ ]:
import pandas as pd
import numpy as np
import os 
import subprocess
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import json
import re
import gcsfs
plt.rcParams['figure.dpi'] = 300

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
google_project_id = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

bucket = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open(f'{bucket}/hypothyroid_data/cb_v2_phenotype_covars.tsv') as f:
    cb = pd.read_csv(f, sep='\t')

In [ ]:
with fs.open(f'{bucket}/hypothyroid_data/huan_phenotype_v4_covars.csv') as f:
    hu = pd.read_csv(f)

In [ ]:
cb_0 = cb[cb['Hypothyroidism']==0].person_id.unique()
cb_1 = cb[cb['Hypothyroidism']==1].person_id.unique()
hu_0 = hu[hu['Hypothyroidism']==0].person_id.unique()
hu_1 = hu[hu['Hypothyroidism']==1].person_id.unique()

In [ ]:
caseoverlap = venn2([set(cb_1), set(hu_1)], set_labels=('Cohort Builder','SQL'))

In [ ]:
venn2([set(cb_0), set(hu_0)], set_labels=('Cohort Builder','BigQuery/SQL'))